# Download NCI THREDDS wind_nowcast pq files

This notebook crawls the NCI THREDDS `wind_nowcast` catalogue, finds `.pq` files in the eight site folders, and downloads them into the project `data/wind_nowcast` directory while preserving the site folder names.

In [ ]:
from pathlib import Path
from urllib.parse import urljoin
import time
import xml.etree.ElementTree as ET

import requests

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

CATALOG_URL = "https://thredds.nci.org.au/thredds/catalog/fx31/publications/wind_nowcast/catalog.xml"
THREDDS_ROOT = "https://thredds.nci.org.au"
CHUNK_SIZE = 1024 * 1024
TIMEOUT = 60

NS = {"thredds": "http://www.unidata.ucar.edu/namespaces/thredds/InvCatalog/v1.0"}
XLINK = "{http://www.w3.org/1999/xlink}href"


def find_project_root(start=None):
    """Find the BOM-Team repo root from the current notebook working directory."""
    current = Path(start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / ".git").exists() or path.name == "BOM-Team":
            return path
    raise RuntimeError(f"Could not find project root from {current}")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "wind_nowcast"
DATA_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT, DATA_DIR

In [ ]:
def fetch_xml(session, url):
    response = session.get(url, timeout=TIMEOUT)
    response.raise_for_status()
    return ET.fromstring(response.content)


def catalogue_refs(root):
    for ref in root.findall(".//thredds:catalogRef", NS):
        href = ref.attrib.get(XLINK)
        title = ref.attrib.get("{http://www.w3.org/1999/xlink}title") or ref.attrib.get("name")
        if href:
            yield title, href


def pq_datasets(root):
    for dataset in root.findall(".//thredds:dataset", NS):
        name = dataset.attrib.get("name", "")
        url_path = dataset.attrib.get("urlPath")
        if url_path and name.endswith(".pq"):
            yield name, url_path


def local_path_from_url_path(url_path):
    marker = "fx31/publications/wind_nowcast/"
    if marker not in url_path:
        raise ValueError(f"Unexpected THREDDS urlPath: {url_path}")
    return DATA_DIR / url_path.split(marker, 1)[1]


def discover_pq_files(catalog_url=CATALOG_URL):
    records = []
    with requests.Session() as session:
        root = fetch_xml(session, catalog_url)
        folder_refs = list(catalogue_refs(root))
        print(f"Found {len(folder_refs)} site folders:", ", ".join(name for name, _ in folder_refs))

        for folder_name, href in folder_refs:
            folder_catalog_url = urljoin(catalog_url, href)
            folder_root = fetch_xml(session, folder_catalog_url)
            for file_name, url_path in pq_datasets(folder_root):
                records.append({
                    "site": folder_name,
                    "name": file_name,
                    "url": urljoin(THREDDS_ROOT, f"/thredds/fileServer/{url_path}"),
                    "local_path": local_path_from_url_path(url_path),
                })

    return records


pq_files = discover_pq_files()
print(f"Found {len(pq_files)} pq files")
pq_files[:5]

In [ ]:
def remote_size(response):
    value = response.headers.get("content-length")
    return int(value) if value and value.isdigit() else None


def download_one(session, record):
    local_path = record["local_path"]
    local_path.parent.mkdir(parents=True, exist_ok=True)

    with session.get(record["url"], stream=True, timeout=TIMEOUT) as response:
        response.raise_for_status()
        size = remote_size(response)

        if local_path.exists() and size is not None and local_path.stat().st_size == size:
            return "skipped"

        temp_path = local_path.with_suffix(local_path.suffix + ".part")
        progress = None
        if tqdm is not None:
            progress = tqdm(total=size, unit="B", unit_scale=True, desc=local_path.name, leave=False)

        try:
            with temp_path.open("wb") as file:
                for chunk in response.iter_content(chunk_size=CHUNK_SIZE):
                    if chunk:
                        file.write(chunk)
                        if progress is not None:
                            progress.update(len(chunk))
            temp_path.replace(local_path)
        finally:
            if progress is not None:
                progress.close()

    return "downloaded"


def download_all(records):
    counts = {"downloaded": 0, "skipped": 0, "failed": 0}
    failures = []

    iterator = tqdm(records, desc="pq files") if tqdm is not None else records
    with requests.Session() as session:
        for record in iterator:
            try:
                status = download_one(session, record)
                counts[status] += 1
            except Exception as exc:
                counts["failed"] += 1
                failures.append((record, exc))
                print(f"Failed: {record['url']} -> {exc}")
            time.sleep(0.05)

    print(counts)
    if failures:
        print("Failed files:")
        for record, exc in failures:
            print(record["url"], exc)
    return counts, failures


counts, failures = download_all(pq_files)

## Merge pq files by site

The merged files are written to `data/wind_nowcast/merged_by_site`, one `.pq` file per site folder.

In [ ]:
try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except ImportError as exc:
    raise ImportError(
        "Merging pq/parquet files requires pyarrow. Install it first with: pip install pyarrow"
    ) from exc


MERGED_DIR = DATA_DIR / "merged_by_site"
MERGED_DIR.mkdir(parents=True, exist_ok=True)


def site_source_dirs(data_dir=DATA_DIR):
    return sorted(
        path for path in data_dir.iterdir()
        if path.is_dir() and path.name != MERGED_DIR.name
    )


def merge_site_pq_files(site_dir, output_dir=MERGED_DIR):
    source_files = sorted(site_dir.glob("*.pq"))
    if not source_files:
        print(f"No pq files found in {site_dir}")
        return None

    output_path = output_dir / f"{site_dir.name}.pq"
    temp_path = output_path.with_suffix(output_path.suffix + ".part")

    writer = None
    rows = 0
    try:
        for file_path in source_files:
            table = pq.read_table(file_path)
            if writer is None:
                writer = pq.ParquetWriter(temp_path, table.schema)
            writer.write_table(table)
            rows += table.num_rows
    finally:
        if writer is not None:
            writer.close()

    temp_path.replace(output_path)
    print(f"Merged {len(source_files)} files -> {output_path} ({rows:,} rows)")
    return output_path


merged_files = []
for site_dir in site_source_dirs():
    merged_path = merge_site_pq_files(site_dir)
    if merged_path is not None:
        merged_files.append(merged_path)

merged_files

## Preview bellambi merged file

Read `bellambi.pq` from the merged output folder and display the first 50 rows and last 50 rows.

In [ ]:
import pandas as pd


bellambi_path = DATA_DIR / "merged_by_site" / "bellambi.pq"
bellambi_df = pd.read_parquet(bellambi_path)

print(f"File: {bellambi_path}")
print(f"Shape: {bellambi_df.shape}")

print("\nFirst 50 rows")
display(bellambi_df.head(50))

print("\nLast 50 rows")
display(bellambi_df.tail(50))